In [0]:
%run ../../common/_utils_config

In [0]:
%run ../../common/_utils_reference_data

In [0]:
%run ../utils/_utils_silver_contracts

In [0]:
%run ../utils/_utils_silver_mappings

In [0]:
%run ../utils/_utils_silver_parser

In [0]:
failures = []


def check(description, condition):
    print(("✅ " if condition else "❌ ") + description)
    if not condition:
        failures.append(description)


def inner_day_counts(df, group_columns, time_column):
    """Rows per group and UTC day, without the first and last day (partial at chunk edges)."""
    daily = df.groupBy(*group_columns, F.to_date(time_column).alias("day")).count()
    days = [r["day"] for r in daily.select("day").distinct().orderBy("day").collect()]
    return daily.filter(F.col("day").isin(days[1:-1]))


contracts = load_all_entity_contracts()
mappings = load_all_mappings()
check("mappings are valid before the dry run", not validate_all_mappings(contracts, mappings))

In [0]:
ree_mapping = mappings["ree__electricity_price"]
ree_df, ree_report = dry_run_mapping(ree_mapping, contracts)
print_dry_run_report(ree_report)

check("REE: rows produced", ree_report["rows"] > 0)
check("REE: no nulls in business keys",
      ree_report["nulls"]["price_type"] == 0 and ree_report["nulls"]["datetime_utc"] == 0)
check("REE: no nulls in resolution_minutes", ree_report["nulls"]["resolution_minutes"] == 0)
check("REE: no failed conversions", not any(ree_report["failed_conversions"].values()))
check("REE: every indicator code is mapped", not ree_report["unmapped_values"])
check("REE: no duplicated PKs", ree_report["duplicated_pks"] == 0)

resolutions = {r["price_type"]: r["resolution_minutes"]
               for r in ree_df.select("price_type", "resolution_minutes").distinct().collect()}
check(f"REE: pvpc is hourly and day_ahead_spot quarter-hourly (found {resolutions})",
      resolutions == {"pvpc": 60, "day_ahead_spot": 15})

misaligned = ree_df.filter(F.expr("(unix_timestamp(datetime_utc) DIV 60) % resolution_minutes != 0")).count()
check(f"REE: every timestamp starts a period of its resolution (misaligned: {misaligned})", misaligned == 0)

ree_days = inner_day_counts(ree_df, ["price_type", "resolution_minutes"], "datetime_utc")
day_counts = {(r["price_type"], r["count"]) for r in ree_days.collect()}
incomplete = ree_days.filter(F.col("count") != F.lit(1440) / F.col("resolution_minutes")).count()
check(f"REE: full UTC days have 1440 / resolution rows (found {sorted(day_counts)})", incomplete == 0)

In [0]:
weather_mapping = mappings["open_meteo_archive__weather_observation_hourly"]
weather_df, weather_report = dry_run_mapping(weather_mapping, contracts)
print_dry_run_report(weather_report)

check("Weather: rows produced", weather_report["rows"] > 0)
check("Weather: no nulls in business keys",
      weather_report["nulls"]["province_code"] == 0 and weather_report["nulls"]["datetime_utc"] == 0)
check("Weather: no failed conversions", not any(weather_report["failed_conversions"].values()))
check("Weather: no duplicated PKs", weather_report["duplicated_pks"] == 0)

provinces = weather_df.select("province_code").distinct().count()
check(f"Weather: 52 provinces (found {provinces})", provinces == 52)

weather_days = inner_day_counts(weather_df, ["province_code"], "datetime_utc")
counts = {r["count"] for r in weather_days.collect()}
check(f"Weather: 24 rows per province and full UTC day (found {sorted(counts)})", counts == {24} or not counts)

# FKs must match the PKs computed independently in Python
location_pks = {key_hash_python([r["province_code"]]) for r in read_reference_csv("locations_es.csv")}
weather_fks = {r[0] for r in weather_df.select("FKLocation").distinct().collect()}
check("Weather: every FKLocation matches a location PK (Python md5)", weather_fks <= location_pks)
data_source_fks = {r[0] for r in weather_df.select("FKDataSource").distinct().collect()}
check("Weather: FKDataSource = md5('open_meteo_archive')",
      data_source_fks == {key_hash_python(["open_meteo_archive"])})

sample = weather_df.select("PKWeatherObservationHourly", "province_code", "datetime_utc", "source_name").first()
expected_pk = key_hash_python([sample["province_code"], sample["datetime_utc"].strftime("%Y-%m-%d %H:%M:%S"),
                               sample["source_name"]])
check("Weather: PK = md5(province_code|datetime_utc|source_name) (Python md5)",
      sample["PKWeatherObservationHourly"] == expected_pk)

In [0]:
if failures:
    raise AssertionError(f"{len(failures)} check(s) failed: {failures}")
print("✅ All silver parser checks passed")